# 🧪 Espacio de pruebas

Cuaderno libre para experimentar **sin tocar el informe**. Nada de lo que se haga aquí
entra en `reports/` salvo que se llame explícitamente a `modelado.exportar()`.

Los datos, los parámetros y las funciones son los mismos que usan los cuatro modelos, así
que cualquier resultado de aquí es comparable con el informe.

**Ideas pendientes** (el trabajo futuro que menciona la sección 8 del informe):

- DBSCAN sobre las mismas variables: ¿aparecen los mismos ejes sin fijar `k`?
- Clustering jerárquico y su dendrograma, para ver si los perfiles anidan.
- Otro `k` en mediocampistas, que es el modelo con la silueta más baja.
- Añadir la temporada 2022-23 y comprobar si los perfiles se mantienen (ARI entre años).

In [1]:
import _bootstrap  # noqa: F401 — pone la raíz del proyecto en sys.path

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import parametros as P
from config import rutas as R
from src import datos, modelado, preparacion, visualizacion
from src.utils import mostrar

# Los mismos datos y los mismos cuatro datasets del informe.
df_base = datos.cargar_base(verbose=False)
datasets = preparacion.construir_datasets(df_base, verbose=False)

print({p: len(datasets[p]) for p in P.POSICIONES})

📂 Cargando datos desde caché local: 'datos_jugadores_permitidos.csv'
{'GK': 137, 'DF': 779, 'MF': 1097, 'FW': 496}


## Punto de partida

`preparar_features` devuelve las tres cosas que necesita cualquier prueba:

| | qué es |
|---|---|
| `X` | matriz escalada — la que entra al modelo |
| `X_90` | los mismos datos sin escalar, en unidades reales por 90 |
| `prep` | imputer, scaler, nombres cortos y la máscara `entrena` |

In [2]:
POS = "MF"   # cambia esto para probar otra posición

X, X_90, prep = preparacion.preparar_features(datasets[POS], P.FEATURES[POS], verbose=False)
entrena = prep["entrena"]

print(f"{P.NOMBRE_POS[POS]}: {X.shape[0]} jugadores × {X.shape[1]} variables "
      f"({entrena.sum()} de entrenamiento)")
X.head()

Mediocampistas: 1097 jugadores × 10 variables (887 de entrenamiento)


,Int/90,TklW/90,Fls/90,Fld/90,CrdY/90,Crs/90,Off/90,Ast/90,G-PK/90,Sh/90
0,1.093200,0.742676,-0.468061,-1.658955,-0.481413,-0.054726,-0.533916,1.053429,0.868799,0.391848
1,0.825974,-0.105177,-0.618805,0.391921,-0.875658,-1.374015,-1.254706,0.910062,-1.298621,-1.536202
2,-0.205347,-1.112376,0.871894,0.436811,1.213338,-1.261382,1.604339,1.185709,1.716397,1.122706
3,-0.716768,-0.618316,-0.764234,-0.202285,-0.765505,0.656756,1.570176,-0.660262,2.106043,1.813482
4,-0.808958,-1.272070,-0.636434,-0.277114,-1.287887,0.257811,-0.208533,1.474312,0.667030,1.016749


## Ejemplo: DBSCAN en vez de K-Means

K-Means impone `k` y clusters convexos del mismo tamaño. DBSCAN no necesita `k` y marca
como ruido a quien no encaja — que en scouting puede ser justo lo interesante.

`eps` es el parámetro sensible: en un espacio escalado a media 0 y desviación 1 con 10
variables, valores entre 1.5 y 3 son un punto de partida razonable.

In [3]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

for eps in [1.5, 2.0, 2.5, 3.0]:
    etiquetas = DBSCAN(eps=eps, min_samples=10).fit_predict(X[entrena])
    n_clusters = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
    ruido = (etiquetas == -1).mean()
    sil = (silhouette_score(X[entrena], etiquetas) if n_clusters > 1 else float("nan"))
    print(f"eps={eps:.1f}  clusters={n_clusters:2d}  ruido={ruido:5.1%}  silueta={sil:.3f}")

eps=1.5  clusters= 1  ruido=98.9%  silueta=nan
eps=2.0  clusters= 1  ruido=20.7%  silueta=nan
eps=2.5  clusters= 1  ruido= 1.0%  silueta=nan
eps=3.0  clusters= 1  ruido= 0.1%  silueta=nan


## Sitio libre

A partir de aquí, lo que haga falta.